# Visualize Rates
### Import libraries

In [1]:
import os
import sys
import importlib
import numpy as np
import pandas as pd

from tqdm.notebook import tqdm

Global paths

In [2]:
# Current Working Directory
CWD = os.getcwd()
# WORKSPACE
WORKSPACE = os.environ['WORKSPACE_PATH']

In [3]:
# PyCOMET
PyCOMET = WORKSPACE + '/ML/PyCOMET/pycomet/'
if importlib.util.find_spec('pycomet') is None:
    sys.path.append(os.path.abspath(PyCOMET))

In [4]:
# SurQCT
SurQCT = WORKSPACE + '/ML/PyCOMET/pycomet/examples/SurQCT/'
sys.path.append(os.path.abspath(SurQCT))

Case-specific paths

In [5]:
case_prefix = WORKSPACE + '/ML/PyCOMET/run/SurQCT/O3_UMN/Inel/configs/'

# Output directory# Output directory
out_dir = CWD + '/Output/VisualizeRates/'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

### Initialize PyCOMET model
Working case

In [6]:
case_dir = case_prefix + '/config.06/run.03/'
sys.path.append(os.path.abspath(case_dir))

In [7]:
os.chdir(case_dir)

Import additional libraries

In [8]:
from pycomet.utilities import utils
from pycomet.apps import functions as fun
# Inputs
import inputs
from inputs.general import flags

Compile model

In [9]:
# Load network architecture
net = fun._net(
    input_mod=inputs
)
# Finalize model
for k in ('train', 'save'):
    flags[k] = 0
flags['load'] = 1
model = fun._model(
    flags=flags,
    input_mod=inputs,
    net=net
)


[PyCOMET]: Initializing 'OpNN' network ...

[PyCOMET]: Building 'OpNN' network ...
Model: "SurQCT_OpNN"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 5)]          0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 5)]          0                                            
__________________________________________________________________________________________________
normalization (Normalization)   (None, 5)            11          input_1[0][0]                    
__________________________________________________________________________________________________
normalization_1 (Normalization) (None, 5)            11          input_2[0][0]                    
____

In [10]:
os.chdir(CWD)

### Initialize SurQCT parameters

Import additional libraries

In [11]:
from surqct.src.chemical_systems import chemical_systems as chem_sys
from surqct.src.rates_type.rates_type import get_rates_type_mod

SurQCT model inputs

In [12]:
surqct_inp = case_prefix + '/data_gen.1/InputData.py'
sys.path.append(os.path.dirname(surqct_inp))

import InputData

Initializing chemical system and rates type module

In [13]:
System = chem_sys.get_system(InputData.SystemName)
print(System.ThermochemicalProc.PathToHDF5File)
# Initializing thermochemical processes
System.ThermochemicalProc.ProcBool = System.ThermochemicalProc.ProcNames == InputData.RatesType
# System.ThermochemicalProc.PathToHDF5File = InputData.PathToHDF5File
System.ThermochemicalProc.init_processes()
# Initializing molecules
for Mol in System.Molecules:
    Mol.read_diatdata()

# Initializing rates type
rates_type_mod = get_rates_type_mod(System)

None


In [14]:
print(System)

Sampling levels

In [15]:
# # Predefined
# iLevels = [1, 10, 32, 34, 35, 81, 82, 98, 1000, 2000, 3000, 4581, 6115]

# # Random
# iLevels = iLevels + list(np.random.choice(NLevels[0], size=10, replace=False) + 1)

# # By rotational quantum number
# JqnVec = [0]
# for jqn in JqnVec:
#     iLevels = iLevels + [idx+1 for idx in np.arange(NLevels[0]) if (DiatData[0]['jqn'].to_numpy(int)[idx] == jqn)]

# InputData.iLevelsVecTest = np.unique(iLevels)

In [16]:
# Predefined
InputData.iLevelsVecTest = [1]#, 98, 3000, 4581, 4599, 6115]

Generate testing data

In [17]:
x_true, y_true = rates_type_mod.generate_testingdata(InputData, System)
data_true = pd.concat([x_true, y_true], axis=1)


[SurQCT]: Generating testing data points ...
[SurQCT]:   Reading Variables ['log_EVib', 'log_ERot', 'log_VMax', 'log_ri', 'log_ro', 'log_ro-ri', 'ri/rMin_min', 'log_ro/rMin_min']
[SurQCT]:   Translational Temperature: T = 1500 K


TypeError: stat: path should be string, bytes, os.PathLike or integer, not NoneType

Generate prediction data

In [ ]:
x_pred, y_pred = rates_type_mod.generate_predictiondata(
    InputData,
    System,
    model.net,
    Labeled=True,
    Columns=inputs.data.columns['x']
)
data_pred = pd.concat([x_pred, y_pred], axis=1)

### Plotting rates

In [ ]:
for T in InputData.TTranVecTest:
    paths = {k: out_dir + f'/T{int(T)}K/{k}/' for k in ('2D','3D')}
    for path in paths.values():
        if not os.path.exists(path):
            os.makedirs(path)
    for iIdx in tqdm(InputData.iLevelsVecTest, desc='Temperature T = ' + str(int(T)) + 'K'):
        iIdx = iIdx-1
        # True values
        TrueData = data_true.loc[(data_true['TTran_i'] == T) & (data_true['Idx_i'] == iIdx) & (data_true['KExcit'] >= InputData.MinRateValue)]
        TrueData = (TrueData['Idx_j'].to_numpy().astype(int), TrueData['KExcit'].to_numpy())
        # Pred values
        PredData = data_pred.loc[(data_pred['TTran_i'] == T) & (data_pred['Idx_i'] == iIdx) & (data_pred['KExcit'] >= InputData.MinRateValue)]
        PredData = (PredData['Idx_j'].to_numpy().astype(int), PredData['KExcit'].to_numpy())
        # Plotting
        rates_type_mod.plot_prediction3D(paths['3D'], iIdx, System, TrueData, PredData)
        plt.show()
        plt.close()
        rates_type_mod.plot_prediction2D(paths['2D'], iIdx, System, TrueData, PredData)
        plt.show()
        plt.close()

In [ ]:
# # Get (v,J) of high value reconstructed endothermic rates

# # Initial state
# iIdx = 4581
# # Final state
# vqnRange = [0, 44]
# jqnRange = [150, 240]

# jIdxVec = data_pred['Idx_j'].to_numpy(dtype=int)
# jqn  = DiatData[1]['jqn'].to_numpy()[jIdxVec].astype(int)
# vqn  = DiatData[1]['vqn'].to_numpy()[jIdxVec].astype(int)

# data_pred_sub = data_pred.loc[(vqnRange[0] <= vqn) & (vqn <= vqnRange[1]) & (jqnRange[0] <= jqn) & (jqn <= jqnRange[1]) & (data_pred['Idx_i'] == iIdx - 1)]
# jIdx = data_pred_sub['Idx_j'].to_numpy(dtype=int)[np.argmax(data_pred_sub['KExcit'])]

# vqn, jqn = DiatData[1][['jqn','vqn']].to_numpy()[jIdx].astype(int)

# print(f"Level {jIdx}: (v, J) = ({vqn}, {jqn})")

Global paths

In [ ]:
# WORKSPACE
WORKSPACE = os.environ['WORKSPACE_PATH']
# PyCOMET
PyCOMET = WORKSPACE + '/ML/PyCOMET/pycomet/'
if importlib.util.find_spec('pycomet') is None:
    sys.path.append(os.path.abspath(PyCOMET))
# SurQCT
SurQCT = WORKSPACE + '/ML/PyCOMET/pycomet/examples/SurQCT/'
sys.path.append(os.path.abspath(SurQCT))
# Current Working Directory
CWD = os.getcwd()

In [ ]:
# Output directory
out_dir = CWD + '/Output/VisualizeRates/'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

### Initialize PyCOMET model
Working case

In [ ]:
case_dir = WORKSPACE + '/ML/PyCOMET/run/SurQCT/mimmo/O3_UMN/KInel/case.01/'
sys.path.append(os.path.abspath(case_dir))

In [ ]:
os.chdir(case_dir)

Import additional libraries

In [ ]:
from pycomet.utilities import utils
from pycomet.apps import functions as fun
# Inputs
import inputs
from inputs.general import flags

Compile model

In [ ]:
# Load network architecture
net = fun._net(
    input_mod=inputs
)
# Finalize model
for k in ('train', 'save'):
    flags[k] = 0
flags['load'] = 1
model = fun._model(
    flags=flags,
    input_mod=inputs,
    net=net
)

In [ ]:
os.chdir(CWD)

### Initialize SurQCT parameters
SurQCT model inputs

In [ ]:
surqct_inp = case_dir + '/data_gen/InputData.py'
sys.path.append(os.path.dirname(surqct_inp))

import InputData

In [ ]:
# Modifiy test input parameters
InputData.MinRateValue = 1.e-20
InputData.TTranVecTest = [1500., 5000., 10000., 20000.]

### Generating data
Import `RatesType` module

In [ ]:
RatesType = __import__(
    f'surqct.src.RatesType.{InputData.NbAtoms}Atoms.{InputData.RatesType}',
    globals(),
    locals(),
    [InputData.RatesType],
    0
)

Read levels data

In [ ]:
LevelsData, DiatData, NLevels = RatesType.read_levelsinfo(InputData, InputData.TTranVecTest)

Sampling levels

In [ ]:
# # Predefined
# iLevels = [1, 10, 32, 34, 35, 81, 82, 98, 1000, 2000, 3000, 4581, 6115]

# # Random
# iLevels = iLevels + list(np.random.choice(NLevels[0], size=10, replace=False) + 1)

# # By rotational quantum number
# JqnVec = [0]
# for jqn in JqnVec:
#     iLevels = iLevels + [idx+1 for idx in np.arange(NLevels[0]) if (DiatData[0]['jqn'].to_numpy(int)[idx] == jqn)]

# InputData.iLevelsVecTest = np.unique(iLevels)

In [ ]:
# Predefined
InputData.iLevelsVecTest = [1, 98, 3000, 4581, 4599, 6115]

Generate testing data

In [ ]:
x_true, y_true = RatesType.generate_testingdata(InputData)
data_true = pd.concat([x_true, y_true], axis=1)

Generate prediction data

In [ ]:
x_pred, y_pred = RatesType.generate_predictiondata(InputData, model.net, Labeled=True)
data_pred = pd.concat([x_pred, y_pred], axis=1)

### Plotting rates

In [ ]:
for T in InputData.TTranVecTest:
    paths = {k: out_dir + f'/T{int(T)}K/{k}/' for k in ('2D','3D')}
    for path in paths.values():
        if not os.path.exists(path):
            os.makedirs(path)
    for iIdx in tqdm(InputData.iLevelsVecTest, desc='Temperature T = ' + str(int(T)) + 'K'):
        iIdx = iIdx-1
        # True values
        TrueData = data_true.loc[(data_true['TTran_i'] == T) & (data_true['Idx_i'] == iIdx) & (data_true['KExcit'] >= InputData.MinRateValue)]
        TrueData = (TrueData['Idx_j'].to_numpy().astype(int), TrueData['KExcit'].to_numpy())
        # Pred values
        PredData = data_pred.loc[(data_pred['TTran_i'] == T) & (data_pred['Idx_i'] == iIdx) & (data_pred['KExcit'] >= InputData.MinRateValue)]
        PredData = (PredData['Idx_j'].to_numpy().astype(int), PredData['KExcit'].to_numpy())
        # Plotting
        RatesType.plot_prediction3D(paths['3D'], iIdx, DiatData, TrueData, PredData)
        plt.show()
        plt.close()
        RatesType.plot_prediction2D(paths['2D'], iIdx, DiatData, TrueData, PredData)
        plt.show()
        plt.close()

In [ ]:
# # Get (v,J) of high value reconstructed endothermic rates

# # Initial state
# iIdx = 4581
# # Final state
# vqnRange = [0, 44]
# jqnRange = [150, 240]

# jIdxVec = data_pred['Idx_j'].to_numpy(dtype=int)
# jqn  = DiatData[1]['jqn'].to_numpy()[jIdxVec].astype(int)
# vqn  = DiatData[1]['vqn'].to_numpy()[jIdxVec].astype(int)

# data_pred_sub = data_pred.loc[(vqnRange[0] <= vqn) & (vqn <= vqnRange[1]) & (jqnRange[0] <= jqn) & (jqn <= jqnRange[1]) & (data_pred['Idx_i'] == iIdx - 1)]
# jIdx = data_pred_sub['Idx_j'].to_numpy(dtype=int)[np.argmax(data_pred_sub['KExcit'])]

# vqn, jqn = DiatData[1][['jqn','vqn']].to_numpy()[jIdx].astype(int)

# print(f"Level {jIdx}: (v, J) = ({vqn}, {jqn})")